In [1]:
# importing
import os
import tempfile
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import json

In [2]:
# Load environment variables in a file called .env
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
if openrouter_api_key:
    print(f"OpenRouter API key exists and begins {openrouter_api_key[:7]}")
else:
    print("OpenRouter API key not set")


OpenAI API Key exists and begins sk-proj-
OpenRouter API key exists and begins sk-or-v


In [3]:
# Clients: OpenAI (GPT) and OpenRouter (Claude)
openai_client = OpenAI(api_key=openai_api_key)

openrouter_url = "https://openrouter.ai/api/v1"
openrouter_client = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

GPT_MODEL = "gpt-4.1-mini"
CLAUDE_MODEL = "anthropic/claude-3.5-sonnet"


In [4]:
# setting system prompt (math tutor)
system_message = """
You are a math tutor. Help the student with mathematics only: arithmetic, algebra, calculus, etc.
Answer concisely and clearly. When you need to compute a numeric expression, use the evaluate_math tool instead of calculating yourself.
Keep your answer under 400 tokens. If the question is not about math, politely say you only help with math.
Respond in markdown format.
"""


In [5]:
# Tool: evaluate_math — safe evaluation of numeric expressions
import ast
import operator

BINARY_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
}
UNARY_OPS = {ast.USub: operator.neg}

def _eval_node(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        left = _eval_node(node.left)
        right = _eval_node(node.right)
        return BINARY_OPS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -_eval_node(node.operand)
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and len(node.args) == 1:
        allowed = {"abs": abs, "round": round, "int": int, "float": float}
        if node.func.id in allowed:
            return allowed[node.func.id](_eval_node(node.args[0]))
    raise ValueError("Unsupported expression")

def evaluate_math(expression: str) -> str:
    """Safely evaluate a numeric math expression (numbers, +, -, *, /, //, **, %, abs, round). Returns result as string or error message."""
    try:
        tree = ast.parse(expression.strip(), mode="eval")
        if not isinstance(tree.body, (ast.BinOp, ast.UnaryOp, ast.Constant, ast.Call)):
            return "Error: only numeric expressions are allowed."
        result = _eval_node(tree.body)
        return str(result)
    except SyntaxError as e:
        return f"Syntax error: {e}"
    except Exception as e:
        return f"Error: {e}"

# Tool definition for OpenAI/OpenRouter API
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "evaluate_math",
            "description": "Evaluate a numeric math expression. Use for arithmetic: +, -, *, /, //, **, %, and functions abs, round. Input must be a single expression, e.g. '2**10 + 3' or 'abs(-5) * 2'.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The math expression to evaluate, e.g. '2**10' or '100 / 3'"},
                },
                "required": ["expression"],
            },
        },
    }
]

In [6]:
# Voice: speech-to-text (Whisper) and text-to-speech (OpenAI TTS) — use OpenAI client
def transcribe_audio(audio_path) -> str:
    """Transcribe audio file to text using OpenAI Whisper. Returns empty string if no file or error."""
    if audio_path is None:
        return ""
    try:
        with open(audio_path, "rb") as f:
            response = openai_client.audio.transcriptions.create(model="whisper-1", file=f)
        return (response.text or "").strip()
    except Exception:
        return ""


def talker(text: str):
    """Convert text to speech using OpenAI TTS. Returns raw bytes (mp3)."""
    if not text or not text.strip():
        return None
    response = openai_client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=text.strip(),
    )
    return response.content


def get_tts_path(text: str):
    """Convert text to speech and save to a temp file. Returns file path or None for Gradio Audio."""
    content = talker(text)
    if content is None:
        return None
    f = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    f.write(content)
    f.close()
    return f.name

In [7]:
# Chat with model choice, tools (evaluate_math), and streaming
def get_client_and_model(model_choice: str):
    """Return (client, model_id) for the chosen UI option."""
    if model_choice == "Claude":
        return openrouter_client, CLAUDE_MODEL
    return openai_client, GPT_MODEL


def _append_user_message(message: str, history: list) -> list:
    if not message or not message.strip():
        return history
    return history + [{"role": "user", "content": message.strip()}]


def handle_tool_calls(message):
    """Handle tool calls from API message (or dict built from stream). Returns list of tool response dicts."""
    tool_calls = message["tool_calls"] if isinstance(message, dict) else message.tool_calls
    responses = []
    for tc in tool_calls:
        if isinstance(tc, dict):
            name = tc["function"]["name"]
            arguments = json.loads(tc["function"]["arguments"])
            tool_call_id = tc["id"]
        else:
            name = tc.function.name
            arguments = json.loads(tc.function.arguments)
            tool_call_id = tc.id
        if name == "evaluate_math":
            content = evaluate_math(arguments.get("expression", ""))
        else:
            content = "Unknown tool"
        # Log so you can see in notebook/console that a tool was called
        print(f"[Tool called] {name}({arguments}) -> {content}", flush=True)
        responses.append({"role": "tool", "content": content, "tool_call_id": tool_call_id})
    return responses


def _stream_reply(history: list, model_choice: str):
    """Stream when model does not call tools; handle tool_calls without streaming (like day4)."""
    client, model_id = get_client_and_model(model_choice)
    messages = [{"role": "system", "content": system_message}] + list(history)
    reply = {"role": "assistant", "content": ""}

    # First request: stream with tools
    stream = client.chat.completions.create(
        model=model_id,
        messages=messages,
        tools=TOOLS,
        stream=True,
    )
    tool_calls_accum = {}  # index -> {id, name, arguments}

    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        finish_reason = chunk.choices[0].finish_reason

        if getattr(delta, "content", None):
            reply["content"] += delta.content
            yield history + [reply], None
        if getattr(delta, "tool_calls", None):
            for tc in delta.tool_calls:
                i = tc.index
                if i not in tool_calls_accum:
                    tool_calls_accum[i] = {"id": "", "name": "", "arguments": ""}
                tool_calls_accum[i]["id"] += tc.id or ""
                tool_calls_accum[i]["name"] += tc.function.name or ""
                tool_calls_accum[i]["arguments"] += tc.function.arguments or ""

        if finish_reason == "tool_calls":
            # Build assistant message from accumulated tool_calls and run tools (no streaming)
            assistant_msg = {
                "role": "assistant",
                "content": reply["content"] or "",
                "tool_calls": [
                    {
                        "id": tool_calls_accum[i]["id"],
                        "type": "function",
                        "function": {
                            "name": tool_calls_accum[i]["name"],
                            "arguments": tool_calls_accum[i]["arguments"],
                        },
                    }
                    for i in sorted(tool_calls_accum.keys())
                ],
            }
            messages.append(assistant_msg)
            messages.extend(handle_tool_calls(assistant_msg))
            # Loop until model returns content (no more tool_calls)
            while True:
                response = client.chat.completions.create(
                    model=model_id,
                    messages=messages,
                    tools=TOOLS,
                    stream=False,
                )
                msg = response.choices[0].message
                if msg.tool_calls:
                    msg_dict = {"role": "assistant", "content": msg.content or "", "tool_calls": msg.tool_calls}
                    messages.append(msg_dict)
                    messages.extend(handle_tool_calls(msg))
                    continue
                reply["content"] = msg.content or ""
                yield history + [reply], get_tts_path(reply["content"])
                return
        elif finish_reason == "stop":
            yield history + [reply], get_tts_path(reply["content"])
            return

    # Stream ended without finish_reason (e.g. some providers); if we have content, already yielded
    if reply["content"]:
        yield history + [reply], get_tts_path(reply["content"])
        return
    # Fallback: no content and no tool_calls accumulated -> yield once
    yield history + [reply], None


In [8]:
# UI: model selector, chat, voice input (mic) and TTS output
with gr.Blocks(title="Math Tutor", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## Math Tutor")
    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=["GPT", "Claude"],
            value="GPT",
            label="Model",
            scale=0,
        )
    chatbot = gr.Chatbot(type="messages", label="Chat", height=400)
    with gr.Row():
        msg = gr.Textbox(placeholder="Ask a math question...", show_label=False, container=False, scale=3)
        audio_input = gr.Audio(sources=["microphone"], type="filepath", label="Voice", scale=0)
    with gr.Row():
        submit_btn = gr.Button("Send", variant="primary")
        voice_btn = gr.Button("Send voice")
        clear_btn = gr.ClearButton([msg, chatbot], value="Clear chat")
    audio_output = gr.Audio(autoplay=True, label="Listen to reply")

    def on_submit(message, history):
        new_history = _append_user_message(message, history or [])
        return "", new_history

    def on_voice_submit(audio_path, history):
        text = transcribe_audio(audio_path)
        if not text:
            return "", history or []
        new_history = _append_user_message(text, history or [])
        return "", new_history

    msg.submit(on_submit, [msg, chatbot], [msg, chatbot], queue=False).then(
        _stream_reply, [chatbot, model_dropdown], [chatbot, audio_output]
    )
    submit_btn.click(on_submit, [msg, chatbot], [msg, chatbot], queue=False).then(
        _stream_reply, [chatbot, model_dropdown], [chatbot, audio_output]
    )
    voice_btn.click(on_voice_submit, [audio_input, chatbot], [msg, chatbot], queue=False).then(
        _stream_reply, [chatbot, model_dropdown], [chatbot, audio_output]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
